In [1]:
import matplotlib.pyplot as plt
import numpy as np
import math
import os
import pandas as pd
from scipy import ndimage as ndi
from skimage.transform import rescale
from skimage.measure import marching_cubes, mesh_surface_area, label, regionprops, regionprops_table
from skimage import util, measure
import napari
from tifffile import imread
from liffile import LifFile
from pathlib import Path
import imagej
from imagej import Mode
import scyjava as sj
from matplotlib.colors import to_rgba

# import warnings
# warnings.filterwarnings("ignore")

ij = imagej.init("sc.fiji:fiji", mode=Mode.INTERACTIVE, add_legacy=True)

IJ = sj.jimport("ij.IJ")
Duplicator = sj.jimport("ij.plugin.Duplicator")
WekaSegmentation = sj.jimport("trainableSegmentation.WekaSegmentation")
ImagePlus = sj.jimport("ij.ImagePlus")
ImageStack = sj.jimport("ij.ImageStack")
FloatProcessor = sj.jimport("ij.process.FloatProcessor")

In [2]:
def numpy_to_imageplus(arr: np.ndarray, title="image") -> ImagePlus:
    if arr.ndim == 2:
        y, x = arr.shape
        pix = np.asarray(arr, dtype=np.float32).ravel()
        java_floats = sj.jarray("f", pix.size)
        # fill java float[]
        for i, v in enumerate(pix.tolist()):
            java_floats[i] = float(v)
        fp = FloatProcessor(x, y, java_floats)
        return ImagePlus(title, fp)

    if arr.ndim == 3:
        z, y, x = arr.shape
        stack = ImageStack(x, y)
        for zi in range(z):
            pix = np.asarray(arr[zi], dtype=np.float32).ravel()
            java_floats = sj.jarray("f", pix.size)
            for i, v in enumerate(pix.tolist()):
                java_floats[i] = float(v)
            fp = FloatProcessor(x, y, java_floats)
            stack.addSlice(fp)
        return ImagePlus(title, stack)

    raise ValueError(f"Unsupported shape: {arr.shape}")

In [3]:
def imageplus_to_numpy(imp: ImagePlus) -> np.ndarray:
    w, h = imp.getWidth(), imp.getHeight()
    n = imp.getStackSize()
    stack = imp.getStack()
    out = []
    for z in range(1, n + 1):
        ip = stack.getProcessor(z)
        pix = np.array(ip.getPixels()).reshape(h, w)
        out.append(pix)
    return np.stack(out, axis=0) if n > 1 else out[0]

In [4]:
def convert(img, target_type_min, target_type_max, target_type):
    """
    Converts an image to a specified data type while scaling its intensity values.

    This function rescales the intensity values of an image from its original range 
    to a new target range specified by `target_type_min` and `target_type_max`, and 
    then converts it to the desired data type.

    This step is required as deconvolved images are not always scaled 0->255! 

    Parameters:
    -----------
    img : numpy.ndarray
        The input image array to be converted.
    target_type_min : int or float
        The minimum value of the target intensity range.
    target_type_max : int or float
        The maximum value of the target intensity range.
    target_type : numpy.dtype
        The desired data type of the output image (e.g., np.uint8, np.float32).

    Returns:
    --------
    new_img : numpy.ndarray
        The rescaled image with values mapped to the new intensity range and converted 
        to the specified data type.

    Notes:
    ------
    - This function performs a linear transformation to scale pixel values.
    - It ensures that the output values are properly mapped between `target_type_min` and 
      `target_type_max`.
    """
    imin = img.min()
    imax = img.max()

    a = (target_type_max - target_type_min) / (imax - imin)
    b = target_type_max - a * imax
    new_img = (a * img + b).astype(target_type)
    return new_img

In [5]:
def apply_weka_with_exact_preprocessing(image: np.ndarray, model_path: str) -> np.ndarray:
    imp = numpy_to_imageplus(image, title="image")
    dup = Duplicator().run(imp)
    IJ.run(dup, "8-bit", "")
    IJ.run(dup, "Auto Threshold", "method=Otsu stack")
    IJ.run(dup, "Erode (3D)", "iso=255")

    try:
        seg = WekaSegmentation(dup)
        seg.loadClassifier(model_path)
        out_imp = seg.applyClassifier(dup, 0, False)
        if out_imp is None:
            out_imp = seg.getClassifiedImage()

        if out_imp is None:
            raise RuntimeError("No classified image returned (out_imp is None). Likely a Java-side error or model/input mismatch.")

        return imageplus_to_numpy(out_imp)

    except:
        # Show Java exception details if present
        print("=== Python/Java exception ===")

In [6]:
def step(axis, xa):
    if axis not in xa.coords or xa.coords[axis].size < 2:
        return None
    return float(xa.coords[axis][1] - xa.coords[axis][0])

In [9]:
def segment_and_quantify(i, lif, lif_path, classifier_path):
    img = lif.images[i]
    filename = "".join(os.path.basename(lif_path).lower().replace(".lif", ""))+"__"+("".join(img.path))
    print(filename)
    try:
        if img.dims == ('T', 'Z', 'Y', 'X'):
            image = img.asarray()
            xa = img.asxarray()

            # liffile coordinates are typically in meters → convert to µm
            x_um = step("X", xa) * 1e6 if step("X", xa) is not None else None
            y_um = step("Y", xa) * 1e6 if step("Y", xa) is not None else None
            z_um = step("Z", xa) * 1e6 if step("Z", xa) is not None else None
            # spacing_um = [x_um, y_um, z_um]

            t0 = image[0,:,:,:]
            t2 = image[2,:,:,:]
            gel_matrix = apply_weka_with_exact_preprocessing(t0, classifier_path)
        
            vasculature_segmentation = (gel_matrix == 0).astype(int)
            vasculature_labels = label(vasculature_segmentation)

            table = regionprops_table(vasculature_labels, properties=('label', 'area'),)

            condition = (table['area'] >= 20)
            input_labels = table['label']
            output_labels = input_labels * condition
            output_labels = util.map_array(vasculature_labels, input_labels, output_labels)
            clean_vasculature_segmentation = output_labels > 0
            # clean_gel_segmentation = (clean_vasculature_segmentation == 0).astype(int)
            
            t2 = convert(t2, 0, 255, np.uint8)
            t0 = convert(t0, 0, 255, np.uint8)

            final_vascular_intensity = np.sum(t2[clean_vasculature_segmentation==1])
            final_gel_intensity = np.sum(t2[clean_vasculature_segmentation==0])
            initial_vascular_intensity = np.sum(t0[clean_vasculature_segmentation==1])
            initial_gel_intensity = np.sum(t0[clean_vasculature_segmentation==0])

            rescaled_t0 = rescale(scale = (z_um/x_um,1,1), image=(t0 ), anti_aliasing = False)
            # rescaled_t2 = rescale(scale = (z_um/x_um,1,1), image=(t2 ), anti_aliasing = False)
            rescaled_vasculature_segmentation = rescale(scale = (z_um/x_um,1,1), image=(clean_vasculature_segmentation ), anti_aliasing = False, order=0, preserve_range=True).astype(clean_vasculature_segmentation.dtype)
            verts, faces, _, _ = marching_cubes(rescaled_vasculature_segmentation.astype(np.uint8), level=0.5, spacing=(x_um,) * 3)
            vasculature_surface_area = mesh_surface_area(verts, faces)
            vascular_volume = np.sum(rescaled_vasculature_segmentation==1) * ((x_um)**3)
            total_volume = rescaled_t0.shape[0] * rescaled_t0.shape[1] * rescaled_t0.shape[2] * ((x_um)**3)
            gel_volume = total_volume - vascular_volume
            p = ((1/360)*(gel_volume)/(vasculature_surface_area))*((np.float64(final_gel_intensity)-np.float64( initial_gel_intensity))/(np.float64(initial_vascular_intensity)-np.float64(initial_gel_intensity)))

            output = pd.DataFrame({
                "filename": [filename],
                "image_shape": [image.shape],
                "final_gel_intensity": [final_gel_intensity],
                "final_vascular_intensity": [final_vascular_intensity],
                "initial_gel_intensity": [initial_gel_intensity],
                "initial_vascular_intensity": [initial_vascular_intensity],
                "vascular_volume_um3": [vascular_volume],
                "gel_volume_um3": [gel_volume],
                "vasculature_surface_area_um2": [vasculature_surface_area],
                "p_um/s": [p],
                "p_cm/s": [p*0.0001],
                })
        else:
            output = pd.DataFrame({
                "filename": [filename],
                "flag": ["IMAGE FAILED"],})
    except:
        output = pd.DataFrame({
            "filename": [filename],
            "flag": ["CZYX Image"],})
    IJ.run("Close All")
    return output


In [11]:
from pathlib import Path
# lif_path = Path(r"C:/Users/taylorhearn/Downloads/2025.10.02_FL33.lif")
classifier_path="C:/Users/taylorhearn/git_repos/image_quantification/Vasculature/classifier.model"
folder = Path(r"C:/Users/taylorhearn/Desktop/3D_permeability")
lif_files = list(folder.glob("*.lif"))

for lif_path in lif_files[-1:]:
    lif_name = "".join(os.path.basename(lif_path).lower().replace(".lif", "")).replace(".", "_")
    print(lif_name)
    all_outputs = []
    with LifFile(lif_path) as lif:
        number_of_lifs = len(lif.images)

        for i in range(number_of_lifs):
            output = segment_and_quantify(i,lif, lif_path, classifier_path)
            all_outputs.append(output)
        totals = pd.concat(all_outputs)
        totals.to_csv("{}.csv".format(lif_name))

m7_2025_12_04_fl39
m7_2025.12.04_fl39__Bead_dev1/P 2
m7_2025.12.04_fl39__Bead_dev1/P 1
m7_2025.12.04_fl39__Bead_dev1/P 3
m7_2025.12.04_fl39__Bead_dev1/P 4
m7_2025.12.04_fl39__Bead_dev1_Merged
m7_2025.12.04_fl39__rLN2_dev1/P 2
m7_2025.12.04_fl39__rLN2_dev1/P 1
m7_2025.12.04_fl39__rLN2_dev1/P 3
m7_2025.12.04_fl39__rLN2_dev1/P 4
m7_2025.12.04_fl39__rLN2_dev1_Merged
m7_2025.12.04_fl39__FL39_LigNeg_dev1/P 4
m7_2025.12.04_fl39__FL39_LigNeg_dev1/P 3
m7_2025.12.04_fl39__FL39_LigNeg_dev1/P 1
m7_2025.12.04_fl39__FL39_LigNeg_dev1/P 2
m7_2025.12.04_fl39__FL39_LigNeg_dev1_Merged
m7_2025.12.04_fl39__FL39_LigPos_dev1/P 1
m7_2025.12.04_fl39__FL39_LigPos_dev1/P 2
m7_2025.12.04_fl39__FL39_LigPos_dev1/P 4
m7_2025.12.04_fl39__FL39_LigPos_dev1/P 3
m7_2025.12.04_fl39__FL39_LigPos_dev1_Merged
m7_2025.12.04_fl39__Bead_dev2/P 1
m7_2025.12.04_fl39__Bead_dev2/P 2
m7_2025.12.04_fl39__Bead_dev2/P 3
m7_2025.12.04_fl39__Bead_dev2/P 4
m7_2025.12.04_fl39__Bead_dev2_Merged
m7_2025.12.04_fl39__rLN2_dev2/P 1
m7_2025.12.0

C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_5612\1537650515.py:37: RuntimeWarning: divide by zero encountered in scalar divide
  a = (target_type_max - target_type_min) / (imax - imin)
C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_5612\1537650515.py:38: RuntimeWarning: invalid value encountered in scalar multiply
  b = target_type_max - a * imax
C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_5612\1537650515.py:39: RuntimeWarning: invalid value encountered in multiply
  new_img = (a * img + b).astype(target_type)
C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_5612\1537650515.py:39: RuntimeWarning: invalid value encountered in cast
  new_img = (a * img + b).astype(target_type)


m7_2025.12.04_fl39__Bead_dev3/P 1


C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_5612\1537650515.py:37: RuntimeWarning: divide by zero encountered in scalar divide
  a = (target_type_max - target_type_min) / (imax - imin)
C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_5612\1537650515.py:38: RuntimeWarning: invalid value encountered in scalar multiply
  b = target_type_max - a * imax
C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_5612\1537650515.py:39: RuntimeWarning: invalid value encountered in multiply
  new_img = (a * img + b).astype(target_type)
C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_5612\1537650515.py:39: RuntimeWarning: invalid value encountered in cast
  new_img = (a * img + b).astype(target_type)


m7_2025.12.04_fl39__Bead_dev3/P 3


C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_5612\1537650515.py:37: RuntimeWarning: divide by zero encountered in scalar divide
  a = (target_type_max - target_type_min) / (imax - imin)
C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_5612\1537650515.py:38: RuntimeWarning: invalid value encountered in scalar multiply
  b = target_type_max - a * imax
C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_5612\1537650515.py:39: RuntimeWarning: invalid value encountered in multiply
  new_img = (a * img + b).astype(target_type)
C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_5612\1537650515.py:39: RuntimeWarning: invalid value encountered in cast
  new_img = (a * img + b).astype(target_type)


m7_2025.12.04_fl39__Bead_dev3/P 4


C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_5612\1537650515.py:37: RuntimeWarning: divide by zero encountered in scalar divide
  a = (target_type_max - target_type_min) / (imax - imin)
C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_5612\1537650515.py:38: RuntimeWarning: invalid value encountered in scalar multiply
  b = target_type_max - a * imax
C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_5612\1537650515.py:39: RuntimeWarning: invalid value encountered in multiply
  new_img = (a * img + b).astype(target_type)
C:\Users\taylorhearn\AppData\Local\Temp\ipykernel_5612\1537650515.py:39: RuntimeWarning: invalid value encountered in cast
  new_img = (a * img + b).astype(target_type)


m7_2025.12.04_fl39__Bead_dev3_Merged
m7_2025.12.04_fl39__rLN2_dev3/P 2
m7_2025.12.04_fl39__rLN2_dev3/P 1
m7_2025.12.04_fl39__rLN2_dev3/P 3
m7_2025.12.04_fl39__rLN2_dev3/P 4
m7_2025.12.04_fl39__rLN2_dev3_Merged
m7_2025.12.04_fl39__FL39_LigNeg_dev3/P 2
m7_2025.12.04_fl39__FL39_LigNeg_dev3/P 1
m7_2025.12.04_fl39__FL39_LigNeg_dev3/P 4
m7_2025.12.04_fl39__FL39_LigNeg_dev3/P 3
m7_2025.12.04_fl39__FL39_LigNeg_dev3_Merged
m7_2025.12.04_fl39__FL39_LigPos_dev3/P 1
m7_2025.12.04_fl39__FL39_LigPos_dev3/P 2
m7_2025.12.04_fl39__FL39_LigPos_dev3/P 3
m7_2025.12.04_fl39__FL39_LigPos_dev3/P 4
m7_2025.12.04_fl39__FL39_LigPos_dev3_Merge
m7_2025.12.04_fl39__Bead_dev4_Merged
m7_2025.12.04_fl39__Bead_dev4/P 2
m7_2025.12.04_fl39__Bead_dev4/P 3
m7_2025.12.04_fl39__Bead_dev4/P 4
m7_2025.12.04_fl39__Bead_dev4/P 1
m7_2025.12.04_fl39__rLN2_dev4/P 2
m7_2025.12.04_fl39__rLN2_dev4/P 1
m7_2025.12.04_fl39__rLN2_dev4/P 3
m7_2025.12.04_fl39__rLN2_dev4/P 4
m7_2025.12.04_fl39__rLN2_dev4_Merged
m7_2025.12.04_fl39__FL39_LigN

In [ ]:
# # path = Path(r"./lif/2025_10_02_FL33.lif")
# lif_path = Path(r"C:/Users/taylorhearn/Downloads/2025.10.02_FL33.lif")
# classifier_path="C:/Users/taylorhearn/git_repos/image_quantification/Vasculature/classifier.model"
# all_outputs = []
# with LifFile(lif_path) as lif:
#     number_of_lifs = len(lif.images)

#     for i in range(number_of_lifs):
#         output = segment_and_quantify(i,lif, lif_path, classifier_path)
#         all_outputs.append(output)

        

In [10]:
totals = pd.concat(all_outputs)
totals

,filename,image_shape,final_gel_intensity,final_vascular_intensity,initial_gel_intensity,initial_vascular_intensity,vascular_volume_um3,gel_volume_um3,vasculature_surface_area_um2,p_um/s,p_cm/s,flag
0,2025.10.02_fl33__Bead_dev1/P 1,"(3, 25, 512, 512)",2909551.0,9850457.0,2105143.0,9671080.0,1.568237e+07,1.540712e+08,1.904569e+06,0.023891,2.389108e-06,NaN
0,2025.10.02_fl33__Bead_dev1/P 2,"(3, 23, 512, 512)",2463866.0,13185374.0,1781222.0,12048073.0,2.490630e+07,1.325015e+08,2.378371e+06,0.010290,1.028953e-06,NaN
0,2025.10.02_fl33__Bead_dev1/P 3,"(3, 27, 512, 512)",3012392.0,18728024.0,1724751.0,13856012.0,3.275091e+07,1.493483e+08,2.982312e+06,0.014765,1.476501e-06,NaN
0,2025.10.02_fl33__Bead_dev1/P 4,"(3, 22, 512, 512)",2906482.0,19207097.0,2228250.0,18392811.0,2.321549e+07,1.249330e+08,1.995130e+06,0.007298,7.298239e-07,NaN
0,2025.10.02_fl33__Bead_dev1_Merged,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IMAGE FAILED
...,...,...,...,...,...,...,...,...,...,...,...,...
0,2025.10.02_fl33__FL33_LPos_dev5/P 1,"(3, 23, 512, 512)",3835385.0,29947461.0,2641418.0,31888206.0,3.026206e+07,1.271458e+08,2.082879e+06,0.006922,6.922284e-07,NaN
0,2025.10.02_fl33__FL33_LPos_dev5/P 2,"(3, 15, 512, 512)",2921541.0,15766614.0,1523057.0,13041086.0,1.723682e+07,8.461529e+07,1.436348e+06,0.019869,1.986854e-06,NaN
0,2025.10.02_fl33__FL33_LPos_dev5/P 3,"(3, 19, 512, 512)",4092354.0,17011647.0,2565664.0,16481147.0,1.595711e+07,1.136729e+08,1.571926e+06,0.022038,2.203812e-06,NaN
0,2025.10.02_fl33__FL33_LPos_dev5/P 4,"(3, 23, 512, 512)",2839026.0,8272715.0,1642476.0,5616798.0,1.233404e+07,1.450738e+08,1.623228e+06,0.074744,7.474369e-06,NaN


In [11]:
totals.to_csv("output_csv.csv")

In [ ]:
viewer = napari.Viewer()
viewer.add_image(t0, name="t0")
viewer.add_image(t2, name="t2")
viewer.add_labels(vasculature_segmentation, name="vasculature_segmentation", colormap={1: np.array(to_rgba("dodgerblue"), dtype=float)})
viewer.add_labels(clean_vasculature_segmentation, name="filtered_labels", colormap={1: np.array(to_rgba("red"), dtype=float)})

c:\Users\taylorhearn\AppData\Local\miniconda3\envs\nap-ij\Lib\site-packages\napari\utils\colormaps\colormap.py:435: UserWarning: color_dict did not provide a default color. Missing keys will be transparent. To provide a default color, use the key `None`, or provide a defaultdict instance.
  warn(
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\nap-ij\Lib\site-packages\napari\utils\colormaps\colormap.py:435: UserWarning: color_dict did not provide a default color. Missing keys will be transparent. To provide a default color, use the key `None`, or provide a defaultdict instance.
  warn(


<Labels layer 'filtered_labels' at 0x152e3398890>

In [ ]:
number_of_lifs = len(LifFile(path).images)
for i in range(number_of_lifs):
    image = LifFile(path).images[i].asarray()
    name = "".join(LifFile(path).images[i].path)
    print(i, name)


0 Bead_dev1/P 1
1 Bead_dev1/P 2
2 Bead_dev1/P 3
3 Bead_dev1/P 4
4 Bead_dev1_Merged
5 rLN1_dev1/P 1
6 rLN1_dev1/P 2
7 rLN1_dev1/P 4
8 rLN1_dev1/P 3
9 rLN1_dev1_Merged
10 FL33_LNeg_dev1/P 3
11 FL33_LNeg_dev1/P 2
12 FL33_LNeg_dev1/P 4
13 FL33_LNeg_dev1/P 1
14 FL33_LNeg_dev1_Merged
15 FL33_LPos_dev1/P 1
16 FL33_LPos_dev1/P 2
17 FL33_LPos_dev1/P 4
18 FL33_LPos_dev1/P 3
19 FL33_LPos_dev1_Merged
20 Bead_dev2/P 1
21 Bead_dev2/P 2
22 Bead_dev2/P 3
23 Bead_dev2/P 4
24 Bead_dev2_Merged
25 rLN1_dev2/P 1
26 rLN1_dev2/P 2
27 rLN1_dev2/P 3
28 rLN1_dev2/P 4
29 rLN1_dev2_Merged
30 FL33_LNeg_dev2/P 1
31 FL33_LNeg_dev2/P 2
32 FL33_LNeg_dev2/P 3
33 FL33_LNeg_dev2/P 4
34 FL33_LNeg_dev2_Merged
35 FL33_LPos_dev2/P 1
36 FL33_LPos_dev2/P 2
37 FL33_LPos_dev2/P 3
38 FL33_LPos_dev2/P 4
39 FL33_LPos_dev2_Merged
40 Bead_dev3/P 1
41 Bead_dev3/P 2
42 Bead_dev3/P 3
43 Bead_dev3/P 4
44 Bead_dev3_Merged
45 rLN1_dev3/P 2
46 rLN1_dev3/P 1
47 rLN1_dev3/P 3
48 rLN1_dev3/P 4
49 rLN1_dev3_Merged
50 FL33_LNeg_dev3/P 1
51 FL33_

In [ ]:
# with LifFile(path) as lif:
#     # list datasets inside the .lif
#     for i, img in enumerate(lif.images):
#         print(i, img.name, img.dims, img.shape)

#     # select by index
#     img = lif.images[0]
#     arr = img.asarray()       # numpy array
#     print(arr.shape, arr.dtype)

Bead_dev1/P 1


In [ ]:
viewer = napari.Viewer()
viewer.add_image(image[0,:,:,:], name="t0")
viewer.add_image(image[-1, :,:, :], name="t2")

<Image layer 't2' at 0x283086b8910>

In [ ]:
viewer = napari.Viewer()


In [ ]:
with LifFile(path) as lif:
    # list datasets inside the .lif
    for i, img in enumerate(lif.images):
        print(i, img.name, img.dims, img.shape)

    # select by index
    img = lif.images[0]
    arr = img.asarray()       # numpy array
    print(arr.shape, arr.dtype)

0 P 1 ('T', 'Z', 'Y', 'X') (3, 25, 512, 512)
1 P 2 ('T', 'Z', 'Y', 'X') (3, 23, 512, 512)
2 P 3 ('T', 'Z', 'Y', 'X') (3, 27, 512, 512)
3 P 4 ('T', 'Z', 'Y', 'X') (3, 22, 512, 512)
4 Bead_dev1_Merged ('C', 'Z', 'Y', 'X') (3, 3, 3286, 2827)
5 P 1 ('T', 'Z', 'Y', 'X') (3, 27, 512, 512)
6 P 2 ('T', 'Z', 'Y', 'X') (3, 26, 512, 512)
7 P 4 ('T', 'Z', 'Y', 'X') (3, 20, 512, 512)
8 P 3 ('T', 'Z', 'Y', 'X') (3, 27, 512, 512)
9 rLN1_dev1_Merged ('C', 'Z', 'Y', 'X') (2, 3, 3289, 2825)
10 P 3 ('T', 'Z', 'Y', 'X') (3, 23, 512, 512)
11 P 2 ('T', 'Z', 'Y', 'X') (3, 22, 512, 512)
12 P 4 ('T', 'Z', 'Y', 'X') (3, 21, 512, 512)
13 P 1 ('T', 'Z', 'Y', 'X') (3, 24, 512, 512)
14 FL33_LNeg_dev1_Merged ('C', 'Z', 'Y', 'X') (2, 3, 3290, 2825)
15 P 1 ('T', 'Z', 'Y', 'X') (3, 23, 512, 512)
16 P 2 ('T', 'Z', 'Y', 'X') (3, 9, 512, 512)
17 P 4 ('T', 'Z', 'Y', 'X') (3, 19, 512, 512)
18 P 3 ('T', 'Z', 'Y', 'X') (3, 15, 512, 512)
19 FL33_LPos_dev1_Merged ('C', 'Z', 'Y', 'X') (2, 3, 3288, 2825)
20 P 1 ('T', 'Z', 'Y', 'X